# 5.3 · multi-hierarchy kv-transfer — is the sub-**sub**-goal carried the same way?

Paper weak point: §5 (NRGS) shifts the hierarchy level, while the rest of the paper fixes it at the sub-goal level; Figure 1 claims *different levels of the hierarchy are maintained the same way* without a direct test.

Here we repeat the **kv-transfer (task-transfer)** setup one level down. With 2 sub-goals:

```
p_src: a'b -> b        p_dst: a'c -> c        kv-transfer target: b
```
Now let the frontier sub-goals be **conjunctions** of sub-sub-goals that share a prefilled first conjunct `x`:
```
p_src: a'(x'y) -> y    p_dst: a'(x'z) -> z    kv-transfer target: y
```
e.g. `words: list[str]` —
`b` = words that start with "b" **and are palindromes**, `c` = words that start with "b" **and have more than 3 characters**;
both prefill `b = [w for w in words if w.startswith("b") and` → the pivot is the **`and`** token.

The intervention is otherwise identical to kv-transfer (`src.diff.functional.run_kv_transfer_intervention`, the paper's multi-predicate config: goal-250 patch/readout heads, α=2, `pad` alignment, no ablation): cache SRC's task-description key+value on the patch heads and transplant it onto DST's. Success = patched DST writes `y` (SRC's sub-sub-goal) instead of `z`. GoalScope read-outs are taken at the `and` pivot (plus, for contrast, at the sub-goal-level `=` of the clean runs). py→py only. Qualitative — no oracle by default.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, re
from dataclasses import dataclass, field
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]

2026-10-17 13:30:48 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Knobs

`SPAN` — which DESCRIPTION span is transplanted (k+v, SRC → DST):
- `"list"` — the whole rendered task list `(1) a … (2) b …` (= the multi-predicate kv-transfer convention, `D.materialize`);
- `"frontier"` — only the frontier sub-goal's description (`the list of words … that start with "b" and are palindromes`);
- `"clause"` — only the distinguishing sub-sub-goal phrase (`are palindromes` → `have more than 3 characters`).

In [2]:
#------------------------------------------------------------------------------------------------------
ALPHA = 2.0                 # OV-write boost of PATCH_HEADS at the pivot on the PATCHED gen/readout (1.0 = off)
ALIGN_KNOB = "pad"          # "pad" = left-pad the shorter span so the dest span is fully overwritten | "right"
SPAN = "list"               # "list" | "frontier" | "clause"   (see above)
PATCH_HEADS = GOAL_HEADS_250
READOUT_HEADS = GOAL_HEADS_250
MAX_NEW = 48
#------------------------------------------------------------------------------------------------------

# ── two-span prompt renderer: full task span (bright_yellow) + the EXACT KV window (orange) ────
# The transplant window is chosen by `align_kv_spans(mt, ..., mode=ALIGN_KNOB)` (src.diff.functional).
def show_spans(tok, title, task_span=None, kv_span=None, piv=-1):
    """Render the prompt exactly as the model sees it. task_span -> bright_yellow (the task text);
    kv_span -> `black on dark_orange` (the exact tokens whose k+v are transplanted); pivot -> salmon.
    Priority pivot > kv > task, so the KV window shows through the task highlight."""
    ids = (prepare_input(prompts=tok, tokenizer=mt) if isinstance(tok, str) else tok)["input_ids"][0].tolist()
    n = len(ids)
    pivpos = n + piv if piv < 0 else piv
    task = set(range(*task_span)) if task_span else set()
    kv = set(range(*kv_span)) if kv_span else set()
    # Mark spans with control-char sentinels and escape the WHOLE joined string once (then swap
    # sentinels for tags). Escaping per-token drops brackets like `[w ...]` / `list[str]` when the
    # tokenizer splits `[` from the following letter across two tokens: the lone `[` isn't escaped,
    # so once joined it reads as a markup tag and rich absorbs it.
    PO, PC, KO, KC, TO, TC = "\x00<", "\x00>", "\x01<", "\x01>", "\x02<", "\x02>"
    parts = []
    for i in range(n):
        s = mt.tokenizer.decode([ids[i]]).replace("\n", "\\n\n")
        if i == pivpos:
            s = PO + s + PC
        elif i in kv:
            s = KO + s + KC
        elif i in task:
            s = TO + s + TC
        parts.append(s)
    txt = escape("".join(parts))
    txt = (txt.replace(PO, "[bold black on salmon1]").replace(PC, "[/bold black on salmon1]")
              .replace(KO, "[bold black on dark_orange]").replace(KC, "[/bold black on dark_orange]")
              .replace(TO, "[bright_yellow]").replace(TC, "[/bright_yellow]"))
    rich_print(Panel(Text.from_markup(txt), title=f"[bold]{escape(title)}[/bold]",
                     title_align="left", border_style="grey42", padding=(1, 1)))

## Instances — a `MultiPredicateInstance` whose frontier prefill runs past `b =` up to the shared conjunct's `and`

`EXAMPLES` holds the hand-written conjunction tasks: a fully-prefilled sub-goal `a`, a shared first conjunct `x` (prefilled), and the two alternative second conjuncts `y` (SRC) / `z` (DST). `kw` are regexes used only for a quick qualitative verdict on the first line the model writes.

In [3]:
@dataclass(frozen=True)
class HierInstance(D.MultiPredicateInstance):
    sub_prefill: str = ""                          # the shared conjunct x, prefilled after `b =` up to `and`
    clauses: dict = field(default_factory=dict)    # frontier id -> its distinguishing sub-sub-goal phrase
    span: str = "list"                             # transplanted span: "list" | "frontier" | "clause"

    def _parts(self, lang, frontier):
        q, pf = super()._parts(lang, frontier)     # letters + var_eq stub -> prefill ends at "    b ="
        return q, pf + " " + self.sub_prefill

    def full_task_text(self, frontier):            # D.materialize reads the k/v span from here
        if self.span == "frontier":
            return frontier.description
        if self.span == "clause":
            return self.clauses[frontier.id]
        return super().full_task_text(frontier)


EXAMPLES = [
    dict(name="words_b", dt="list[str]", var="words",
         a=("the number of words", "{res} = len({arg})"),
         template='the list of words in `words` that {x} and {c}', x='start with "b"',
         prefill='[w for w in words if w.startswith("b") and',
         y=dict(id="palindrome", nl="are palindromes", kw=r"\[::-1\]"),
         z=dict(id="len_gt3", nl="have more than 3 characters", kw=r"len\(w\)\s*>")),
    dict(name="words_long", dt="list[str]", var="words",
         a=("the number of distinct words", "{res} = len(set({arg}))"),
         template='the list of words in `words` that {x} and {c}', x="have more than 3 characters",
         prefill="[w for w in words if len(w) > 3 and",
         y=dict(id="upper", nl="are all uppercase", kw=r"isupper\(\)"),
         z=dict(id="has_e", nl='contain the letter "e"', kw=r"['\"]e['\"]\s+in")),
    dict(name="nums_even", dt="list[int]", var="nums",
         a=("the sum of the numbers", "{res} = sum({arg})"),
         template="the list of numbers in `nums` that {x} and {c}", x="are even",
         prefill="[x for x in nums if x % 2 == 0 and",
         y=dict(id="gt10", nl="are greater than 10", kw=r">\s*10"),
         z=dict(id="neg", nl="are negative", kw=r"<\s*0")),
    dict(name="chars_alpha", dt="str", var="s",
         a=("the length of the string", "{res} = len({arg})"),
         template="the list of characters in `s` that {x} and {c}", x="are letters",
         prefill="[ch for ch in s if ch.isalpha() and",
         y=dict(id="upper", nl="are uppercase", kw=r"isupper\(\)"),
         z=dict(id="vowel", nl="are vowels", kw=r"aeiou")),
    dict(name="int_even", dt="int", var="n",
         a=("the absolute value of the number", "{res} = abs({arg})"),
         template="whether the number is {x} and {c}", x="even",
         prefill="n % 2 == 0 and",
         y=dict(id="pos", nl="positive", kw=r"n\s*>\s*0"),
         z=dict(id="div3", nl="divisible by 3", kw=r"%\s*3")),
]


def make_instance(ex, flip=False, span=None):
    # SRC pursues `y`, DST pursues `z` (swapped with flip=True)
    y, z = (ex["z"], ex["y"]) if flip else (ex["y"], ex["z"])
    front = lambda c: D.Predicate(id=c["id"], description=ex["template"].format(x=ex["x"], c=c["nl"]),
                                  result_name=c["id"], impl={"python": "", "cpp": ""})
    src_f, dst_f = front(y), front(z)
    a = D.Predicate(id="a", description=ex["a"][0], result_name="a", impl={"python": ex["a"][1], "cpp": ""})
    return HierInstance(
        datatype=ex["dt"], shared=(a,), src_frontier=src_f, dst_frontier=dst_f, var_name=ex["var"],
        presentation="listed", src_lang="python", dst_lang="python", naming="letters", letter_stub="var_eq",
        meta=dict(datatype=ex["dt"], example=ex["name"], flip=flip),
        sub_prefill=ex["prefill"], clauses={src_f.id: y["nl"], dst_f.id: z["nl"]}, span=span or SPAN,
    ), y, z

## Render one instance — SRC/DST prompts, the transplanted span (orange) and the `and` pivot

In [4]:
EX_IDX, FLIP = 0, False

inst, Y, Z = make_instance(EXAMPLES[EX_IDX], flip=FLIP)
m = D.materialize(inst)
S0, T0 = token_span(m["src_prompt"], m["src_task_text"]), token_span(m["dst_prompt"], m["dst_task_text"])
src_tok, dst_tok, SRC_POS, DST_POS, SRC_SPAN, DST_SPAN = align_kv_spans(
    mt, m["src_prompt"], m["dst_prompt"], S0, T0, mode=ALIGN_KNOB,
    src_text=m["src_task_text"], dst_text=m["dst_task_text"])
for name, tok, span, pos in [("SRC", src_tok, SRC_SPAN, SRC_POS), ("DST", dst_tok, DST_SPAN, DST_POS)]:
    last = mt.tokenizer.decode(tok["input_ids"][0, -1])
    show_spans(tok, f"{name} · span={SPAN} {tuple(span)} · pivot token {last!r}",
               task_span=span, kv_span=(pos[0], pos[-1] + 1), piv=-1)
    assert last.strip() == "and", f"{name} pivot is {last!r}, expected 'and'"
print(f"SRC sub-sub-goal y = {Y['nl']!r}   |   DST sub-sub-goal z = {Z['nl']!r}")

╭─ SRC · span=list (63, 95) · pivot token ' and' ─────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ (1) the number of words\n                                                                                       │
│ (2) the list of words in `words` that start with "b" and are palindromes\n                                      │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(words)\n                                                                                            │
│     b = [w for w in words if w.startswith("b") and                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · span=list (63, 95) · pivot token ' and' ─────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words\n                                                                                       │
│ (2) the list of words in `words` that start with "b" and have more than 3 characters\n                          │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = len(words)\n                                                                                            │
│     b = [w for w in words if w.startswith("b") and                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

SRC sub-sub-goal y = 'are palindromes'   |   DST sub-sub-goal z = 'have more than 3 characters'


## Intervention — transplant SRC's description k+v onto DST, read out at `and`

`verdict` is a regex check on the first line the patched model writes (`y` = SRC's sub-sub-goal = success, `z` = DST's native = no-op). The extra GoalScope read-outs at the clean runs' `=` token show what the goal heads carry one level up (the whole sub-goal `b`), for contrast with the `and` read-out (the next sub-sub-goal).

In [5]:
from src.diff.functional import run_kv_transfer_intervention


def eq_pos(prompt):
    # index of the sub-goal-level `=` (the frontier's `b =`): the last '=' token before the pivot
    ids = prepare_input(prompts=prompt, tokenizer=mt)["input_ids"][0].tolist()
    return max(i for i, t in enumerate(ids) if mt.tokenizer.decode([t]).strip() == "=")


def verdict(gen, y, z):
    first = gen.strip().split("\n")[0]
    hy, hz = bool(re.search(y["kw"], first)), bool(re.search(z["kw"], first))
    return "y (SRC)" if hy and not hz else "z (DST)" if hz and not hy else "both" if hy else "neither"


def run_one(ex, flip=False, span=None, verbose=True, eq_readout=True):
    inst, y, z = make_instance(ex, flip=flip, span=span)
    out = run_kv_transfer_intervention(mt=mt, inst=inst, patch_heads=PATCH_HEADS, readout_heads=READOUT_HEADS,
                                       max_new=MAX_NEW, alpha=ALPHA, align=ALIGN_KNOB)
    v = {k: verdict(out[f"gen_{k}"], y, z) for k in ("src", "dst", "patched")}
    if verbose:
        rich_print(f"[bold]{ex['name']}{' (flipped)' if flip else ''} · span={inst.span}[/bold]   "
                   f"y(SRC)={y['nl']!r}  z(DST)={z['nl']!r}")
        rich_print(f"  [red1]clean SRC[/red1]   [{v['src']}] : {escape(out['gen_src'].split(chr(10))[0])}")
        rich_print(f"  [sea_green3]clean DST[/sea_green3]   [{v['dst']}] : {escape(out['gen_dst'].split(chr(10))[0])}")
        rich_print(f"  [salmon1]patched DST[/salmon1] [{v['patched']}] : {escape(out['gen_patched'].split(chr(10))[0])}")
        rich_print(f"  goalscope @ `and` · SRC    : [italic]{escape(out['ro_src'])}[/italic]")
        rich_print(f"  goalscope @ `and` · DST    : [italic]{escape(out['ro_dst'])}[/italic]")
        rich_print(f"  goalscope @ `and` · patched: [italic]{escape(out['ro_patched'])}[/italic]")
        if eq_readout:
            for side, pr in (("SRC", out["src_prompt"]), ("DST", out["dst_prompt"])):
                goalscope_readout(f"  goalscope @ `=`   · {side}   ", pr, eq_pos(pr), heads=READOUT_HEADS)
    return dict(example=ex["name"], flip=flip, span=inst.span, **{f"verdict_{k}": x for k, x in v.items()}, **out)


res = run_one(EXAMPLES[EX_IDX], flip=FLIP)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=132 ' and' => I'm checking if the string is a palindrome by comparing it to its reverse.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=132 ' and' => I'm checking if the length of the string is greater than 4.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

... (8 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope @ `=`   · SRC     pos=118 ' =' => I'm filtering for strings that are palindromes and checking if they 
are longer than 3 characters.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

  goalscope @ `=`   · DST     pos=119 ' =' => I'm filtering a list to include only strings longer than 5 
characters.

## All examples × both directions (compact)

In [6]:
# rows = []
# for ex in EXAMPLES:
#     for flip in (False, True):
#         rows.append(run_one(ex, flip=flip, eq_readout=False))

# from rich.table import Table
# t = Table(title=f"multi-hierarchy kv-transfer · span={SPAN} · α={ALPHA} · {len(PATCH_HEADS)} heads")
# for c in ("example", "SRC→y", "clean SRC", "clean DST", "patched DST", "goalscope@and · patched"):
#     t.add_column(c)
# for r in rows:
#     ex = next(e for e in EXAMPLES if e["name"] == r["example"])
#     y = (ex["z"] if r["flip"] else ex["y"])["nl"]
#     t.add_row(r["example"], y, r["verdict_src"], r["verdict_dst"], r["verdict_patched"], escape(r["ro_patched"][:80]))
# rich_print(t)
# print(f"patched -> y (SRC): {sum(r['verdict_patched'] == 'y (SRC)' for r in rows)}/{len(rows)}")

## Span ablation — transplant the whole list vs only the frontier description vs only the sub-sub-goal clause

In [7]:
for span in ("list", "frontier", "clause"):
    run_one(EXAMPLES[EX_IDX], flip=FLIP, span=span, eq_readout=False)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=132 ' and' => I'm checking if the string is a palindrome by comparing it to its reverse.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=132 ' and' => I'm checking if the length of the string is greater than 4.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=132 ' and' => I'm checking if the current string is a palindrome and reads the same forwards and backwards.

... (4 outputs left out)

goalscope @ `and` · SRC    : I'm checking if the string is a palindrome by comparing it to its reverse.

goalscope @ `and` · DST    : I'm checking if the length of the string is greater than 4.

goalscope @ `and` · patched: I'm checking if the current string is a palindrome and reads the same forwards and 
backwards.

KeyboardInterrupt: 

## Sampled instances — `src.diff.hier_data` (the batch-eval dataloader, `scripts/diff/hier_kv_eval.py`)

Conditions come from the filter bank (`D.FILTER_TASKS`, `list[str]` / `list[int]`); `a` is a q-bank sub-goal that overlaps none of x / y / z. Notation (the eval's SRC target is `z`, DST's native is `y`):
```
p_dst      : a'(x'y) -> y
p_src_match: a'(x'z) -> z      H_ORDER = "match"
p_src_diff : a'(zx') -> z      H_ORDER = "diff"   (SRC spec lists z before x; the code still writes x first)
```
`H_CONN` = `"and"` | `"or"` (same connective on both sides). The same seed gives the same (a, x, y, z) for both orders. Rule verdict (`H.rule_verdict`) executes the first written condition on a pool: `src` = z, `dst` = y, `x_only`, `neither`, `error`.

**py→cpp** (`H_DST_LANG = "cpp"`; SRC stays Python, its task-list k+v is transplanted as before): the DST frontier becomes a C++ range-for filter
```
    std::vector<std::string> b;
    for (const auto& w : words) if (!w.empty() && w.front() == 'a' &&        <- pivot " &&" (" ||" for or)
```
Emptiness guards (`!w.empty() &&`) of any of x / y / z are prefilled before x; a compound x (ranges) is parenthesised; `perfect_square` / `is_prime` (lambda conditions) are excluded. The rule verdict is Python-only (`na` for C++) — judge with the oracle.

In [ ]:
import src.diff.hier_data as H
from src.diff.eval import judge_causal, eval_readouts

H_DT, H_CONN, H_ORDER, H_N, H_SEED, H_IDX = "list[str]", "and", "match", 3, 1123, 1   # H_SEED = the batch eval's base seed
H_DST_LANG = "cpp"   # "python" (py->py) | "cpp" (py->cpp)
H_SRC_LANG = "python"

# the batch eval's sample (scripts/diff/hier_kv_eval.py): instance i here = instance i there (prefix-stable in n)
hsetups = H.seeded_setups(H_DT, H_CONN, H_ORDER, H_SRC_LANG, H_DST_LANG, n_samples=32, seed=H_SEED)[0][:H_N]
for i, s in enumerate(hsetups):
    mm = s.meta
    print(f"[{i}] a={mm['a']:<24} x={mm['x']:<16} y(DST)={mm['y']:<16} z(SRC)={mm['z']}")

hinst = hsetups[H_IDX]
hm = D.materialize(hinst)
_s0, _t0 = token_span(hm["src_prompt"], hm["src_task_text"]), token_span(hm["dst_prompt"], hm["dst_task_text"])
h_src_tok, h_dst_tok, H_SRC_POS, H_DST_POS, H_SRC_SPAN, H_DST_SPAN = align_kv_spans(
    mt, hm["src_prompt"], hm["dst_prompt"], _s0, _t0, mode=ALIGN_KNOB,
    src_text=hm["src_task_text"], dst_text=hm["dst_task_text"])
show_spans(h_src_tok, f"SRC · {H_ORDER} · z={hinst.meta['z']}", task_span=H_SRC_SPAN,
           kv_span=(H_SRC_POS[0], H_SRC_POS[-1] + 1), piv=-1)
show_spans(h_dst_tok, f"DST ({H_DST_LANG}) · y={hinst.meta['y']}", task_span=H_DST_SPAN,
           kv_span=(H_DST_POS[0], H_DST_POS[-1] + 1), piv=-1)

[0] a=count_capitalized        x=starts_a         y(DST)=same_reversed    z(SRC)=short
[1] a=count_all_lowercase      x=starts_a         y(DST)=last_is_digit    z(SRC)=same_reversed
[2] a=count_starts_vowel       x=same_reversed    y(DST)=capitalized      z(SRC)=all_alpha


╭─ SRC · match · z=same_reversed ─────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words written entirely in lowercase letters\n                                                 │
│ (2) the list of words in `words` that start with the letter a and are unchanged when reversed\n                 │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w.islower())\n                                                                  │
│     b = [w for w in words if w.startswith('a') and                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST (cpp) · y=last_is_digit ───────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a C++ program that implements a function `func` which, given a list of strings `words`, computes:\n       │
│ (1) the number of words written entirely in lowercase letters\n                                                 │
│ (2) the list of words in `words` that start with the letter a and end with a digit\n                            │
│ Assume that all the libraries have been already imported.\n                                                     │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ auto func(std::vector<std::string> words){\n                                                                    │
│     auto a = 0;\n                                                                                               │
│     for(auto &a_w : words){ bool a_has = false, a_ok = true; for(char a_c : a_w){ if(std::isalpha((unsigned     │
│ char)a_c)){ a_has = true; if(std::isupper((unsigned char)a_c)) a_ok = false; } } if(a_has && a_ok) a++; }\n     │
│     std::vector<std::string> b;\n                                                                               │
│     for (const auto& w : words) if (!w.empty() && w.front() == 'a' &&                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [ ]:
def run_hier(inst, oracle=False, verbose=True):
    out = run_kv_transfer_intervention(mt=mt, inst=inst, patch_heads=PATCH_HEADS, readout_heads=READOUT_HEADS,
                                       max_new=MAX_NEW, alpha=ALPHA, align=ALIGN_KNOB)
    rule = {k: H.rule_verdict(inst, out[f"gen_{k}"])["label"] for k in ("src", "dst", "patched")}
    causal = judge_causal(inst, out["gen_patched"]) if oracle else None
    # sanity: the CLEAN runs should judge as their own sub-sub-goal (clean SRC -> src/z, clean DST -> dst/y)
    clean = {k: judge_causal(inst, out[f"gen_{k}"])["label"] for k in ("src", "dst")} if oracle else None
    readout = eval_readouts(inst, out["ro_src"], out["ro_dst"], out["ro_patched"]) if oracle else None
    if verbose:
        mm = inst.meta
        rich_print(f"[bold]{mm['datatype']} · {mm['connective']} · {mm['order']} · py->{mm.get('dst_lang', 'python')}[/bold]  a={mm['a']} x={mm['x']}  "
                   f"z(SRC)={mm['z']!r}  y(DST)={mm['y']!r}")
        for k, col in (("src", "red1"), ("dst", "sea_green3"), ("patched", "salmon1")):
            rich_print(f"  [{col}]{k:<8}[/{col}] rule={rule[k]:<8}: {escape(out[f'gen_{k}'].split(chr(10))[0])}")
        for k in ("src", "dst", "patched"):
            rich_print(f"  goalscope @ `{mm['connective']}` · {k:<8}: [italic]{escape(out[f'ro_{k}'])}[/italic]")
        if oracle:
            rich_print(f"  [bold]oracle[/bold] clean SRC={clean['src']} (want src) · clean DST={clean['dst']} (want dst) · "
                       f"patched causal={causal['label']} (triggered={causal['triggered']}) · "
                       f"readout {readout['source']}>{readout['dest']}>{readout['patched']} "
                       f"(valid={readout['valid']}, shifted={readout['shifted']})")
    return dict(meta=dict(inst.meta), rule=rule, causal=causal, clean_oracle=clean, readout=readout, **out)


hres = run_hier(hinst, oracle=True)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=144 ' and' => I'm checking if the string is equal to its reverse (palindrome check).

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=242 ' &&' => I'm checking if the last character of a string is a digit.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=242 ' &&' => Sure, please provide the line of code you want me to check.

... (4 outputs left out)

goalscope @ `and` · src     : I'm checking if the string is equal to its reverse (palindrome check).

goalscope @ `and` · dst     : I'm checking if the last character of a string is a digit.

goalscope @ `and` · patched : Sure, please provide the line of code you want me to check.

oracle clean SRC=src (want src) · clean DST=dst (want dst) · patched causal=src (triggered=True) · readout 
src>dst>neither (valid=True, shifted=False)

In [ ]:
# both orders on the SAME sampled (a, x, y, z). Rule verdict is python-only -> for H_DST_LANG='cpp' set ORACLE=True
ORACLE = H_DST_LANG == "cpp"
hrows = []
for order in ("match", "diff"):
    for inst in H.seeded_setups(H_DT, H_CONN, order, H_SRC_LANG, H_DST_LANG, n_samples=32, seed=H_SEED)[0][:H_N]:
        hrows.append(run_hier(inst, oracle=ORACLE, verbose=False))

from rich.table import Table
t = Table(title=escape(f"hier kv-transfer · py->{H_DST_LANG} · {H_DT} · {H_CONN} · α={ALPHA} · {len(PATCH_HEADS)} heads"))
for c in ("order", "x", "z (SRC)", "y (DST)", "rule: clean SRC", "rule: clean DST", "rule: patched",
          "oracle: clean SRC", "oracle: clean DST", "oracle: patched", f"goalscope@{H_CONN} · patched"):
    t.add_column(c)
for r in hrows:
    mm = r["meta"]
    co = r["clean_oracle"] or {}
    t.add_row(mm["order"], mm["x"], mm["z"], mm["y"], r["rule"]["src"], r["rule"]["dst"], r["rule"]["patched"],
              co.get("src", "-"), co.get("dst", "-"), r["causal"]["label"] if r["causal"] else "-",
              escape(r["ro_patched"][:70]))
rich_print(t)
for order in ("match", "diff"):
    rs = [r for r in hrows if r["meta"]["order"] == order]
    msg = f"{order}: patched -> z (rule) {sum(r['rule']['patched'] == 'src' for r in rs)}/{len(rs)}"
    if ORACLE:
        ok = [r for r in rs if r["clean_oracle"]["src"] == "src" and r["clean_oracle"]["dst"] == "dst"]
        msg += (f" · oracle: clean runs valid {len(ok)}/{len(rs)}, patched -> z {sum(r['causal']['triggered'] for r in rs)}/{len(rs)}"
                f" (among valid {sum(r['causal']['triggered'] for r in ok)}/{len(ok)})")
    print(msg)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=144 ' and' => I'm checking if the length of the string is less than 4.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=196 ' &&' => Sure, please provide the line of code you want me to check.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=196 ' &&' => I'm checking if the string length is less than 4.

... (28 outputs left out)

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=236 ' &&' => I'm checking if the string contains only alphabetic characters.

                         hier kv-transfer · py->cpp · list[str] · and · α=2.0 · 250 heads                          
┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓
┃       ┃          ┃          ┃         ┃ rule:    ┃ rule:   ┃          ┃ oracle: ┃ oracle:  ┃         ┃ goalsco… ┃
┃       ┃          ┃          ┃         ┃ clean    ┃ clean   ┃ rule:    ┃ clean   ┃ clean    ┃ oracle: ┃ ·        ┃
┃ order ┃ x        ┃ z (SRC)  ┃ y (DST) ┃ SRC      ┃ DST     ┃ patched  ┃ SRC     ┃ DST      ┃ patched ┃ patched  ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩
│ match │ starts_a │ short    │ same_r… │ na       │ na      │ na       │ src     │ dst      │ src     │ I'm      │
│       │          │          │         │          │         │          │         │          │         │ checking │
│       │          │          │         │          │         │          │         │          │         │ if the   │
│       │          │          │         │          │         │          │         │          │         │ string   │
│       │          │          │         │          │         │          │         │          │         │ length   │
│       │          │          │         │          │         │          │         │          │         │ is less  │
│       │          │          │         │          │         │          │         │          │         │ than 4.  │
│ match │ starts_a │ same_re… │ last_i… │ na       │ na      │ na       │ src     │ dst      │ src     │ Sure,    │
│       │          │          │         │          │         │          │         │          │         │ please   │
│       │          │          │         │          │         │          │         │          │         │ provide  │
│       │          │          │         │          │         │          │         │          │         │ the line │
│       │          │          │         │          │         │          │         │          │         │ of code  │
│       │          │          │         │          │         │          │         │          │         │ you want │
│       │          │          │         │          │         │          │         │          │         │ me to    │
│       │          │          │         │          │         │          │         │          │         │ check.   │
│ match │ same_re… │ all_alp… │ capita… │ na       │ na      │ na       │ src     │ dst      │ src     │ I'm      │
│       │          │          │         │          │         │          │         │          │         │ checking │
│       │          │          │         │          │         │          │         │          │         │ if the   │
│       │          │          │         │          │         │          │         │          │         │ string   │
│       │          │          │         │          │         │          │         │          │         │ consists │
│       │          │          │         │          │         │          │         │          │         │ only of  │
│       │          │          │         │          │         │          │         │          │         │ alphabe… │
│       │          │          │         │          │         │          │         │          │         │ charact… │
│ diff  │ starts_a │ short    │ same_r… │ na       │ na      │ na       │ src     │ dst      │ src     │ I'm      │
│       │          │          │         │          │         │          │         │          │         │ checking │
│       │          │          │         │          │         │          │         │          │         │ if the   │
│       │          │          │         │          │         │          │         │          │         │ string   │
│       │          │          │         │          │         │          │         │          │         │ is less  │
│       │          │          │         │          │    

match: patched -> z (rule) 0/3 · oracle: clean runs valid 3/3, patched -> z 3/3 (among valid 3/3)
diff: patched -> z (rule) 0/3 · oracle: clean runs valid 3/3, patched -> z 3/3 (among valid 3/3)


## Path patching — what the goal heads read at the connective pivot

The path-patching setup of `05_kv_transfer.ipynb` (report `agent_reports/claude/analysis/path-patching-report.md`), one level
down. Same functions as the batch eval (`src.diff.path_patch`: `sample_hier_instances` → `build_geometry` →
`intervention_stack` / `clean_runs` / `run_patched` / `res_readout` → `judge`), so everything here matches what a batch run
would compute.

```
p_src: a'(x' z) -> z      k/v donor (I1, I3)            pivot = the connective (` and` / ` or`; ` &&` / ` ||` in C++)
p_dst: a'(x' y) -> y      the patched run
p_res: a'(x' e) -> e      I2 residual donor: a 4th condition e ∉ {x, y, z}, compatible with them, x<conn>e observably
                          different from x<conn>y and x<conn>z, rendered in DST's language with a string-identical prefill
```
* **I1** `TASK_PATCH_HEADS` read SRC's window k/v (query rule `PH_I1_QUERY`); **I2** every layer's output residual on
  `<|im_end|> … x` (pivot excluded) ← the donor (`PH_I2_DONOR`: p_res | clean DST); **I3** `GOAL_PATCH_HEADS` read SRC's window
  k/v from the pivot on, optionally values × `PH_I3_V_SCALE`, OV boost `PH_ALPHA`.
* **Window** `PH_WINDOW`: the task span (`PH_SPAN`: whole task list | frontier item) or task span + progress (A1 `\n`-padding).
* Eval = the batch oracle (`PP.judge`: causal on the first written condition, c-vs-d readout, 3-way {d, c, e} with the p_res
  donor; BOTH is a miss) + `H.rule_verdict` (execution of the first written condition, Python destinations only).

In [8]:
#------------------------------------------------------------------------------------------------------
PH_DT, PH_CONN, PH_ORDER = "list[str]", "and", "match"      # datatype · connective · SRC spec order (match | diff)
PH_SRC_LANG, PH_DST_LANG = "python", "python"
PH_SEED, PH_IDX = 1123, 0                                   # the batch eval's sample: PP.sample_hier_instances(seed=PH_SEED)[PH_IDX]
PH_SPAN = "list"                # task span: "list" (whole task list) | "frontier" (the frontier item)
PH_WINDOW = "span"              # what I1 / I3 transplant: "span" | "span+progress" (task span .. pivot, A1-padded)
PH_INCLUDE_PIVOT = True         # span+progress: also transplant the pivot itself
# I1 — task heads read SRC's window k/v
PH_I1 = False
TASK_PATCH_HEADS = GOAL_HEADS_250
PH_I1_QUERY = "all"
# I2 — restore the progress <|im_end|> .. x (pivot excluded) from the donor
PH_I2 = True
PH_I2_FROM = "im_end"           # "im_end" | "span_end"
PH_I2_LAYERS = "all"
PH_I2_DONOR = "res"             # "res" = p_res a'(x' e) | "dst" = clean DST
PH_RES_FRONTIER = None          # e's filter-condition id; None = seeded pick (as the batch eval)
# I3 — goal heads read SRC's window k/v from the pivot on
PH_I3 = True
GOAL_PATCH_HEADS = GOAL_HEADS_250
PH_I3_QUERY = "last"
PH_I3_V_SCALE = 1.0             # x the transplanted SRC window VALUES read by GOAL_PATCH_HEADS (1.0 = off)
PH_ALPHA = 1.0                  # OV boost of GOAL_PATCH_HEADS at -1 (1.0 = off)
PH_READOUT_HEADS = GOAL_HEADS_250
PH_MAX_NEW = 48
PH_ORACLE = True
#------------------------------------------------------------------------------------------------------
import src.diff.path_patch as PP
import src.diff.hier_data as H
from types import SimpleNamespace
from rich.table import Table
from rich.console import Console
PH_I2_LAYER_LIST = list(range(mt.n_layer)) if PH_I2_LAYERS == "all" else sorted(PH_I2_LAYERS)
PH_CTX = SimpleNamespace(mt=mt, group=P.GROUP, readout_heads=PH_READOUT_HEADS, gsp=getattr(P, "gsp", None))
PH_KW = dict(task_heads=TASK_PATCH_HEADS, goal_heads=GOAL_PATCH_HEADS, i1=PH_I1, i2=PH_I2, i3=PH_I3, i1_query=PH_I1_QUERY,
             i3_query=PH_I3_QUERY, v_scale=PH_I3_V_SCALE, alpha=PH_ALPHA, donor=PH_I2_DONOR, i2_layers=PH_I2_LAYER_LIST,
             ctx=PH_CTX)

In [9]:
# ── instance (the batch eval's sampler) + geometry (PP.build_geometry) ──
ph_insts, _ = PP.sample_hier_instances(PH_DT, PH_CONN, PH_ORDER, PH_SRC_LANG, PH_DST_LANG, n_samples=32, seed=PH_SEED)
for i, s in enumerate(ph_insts[: max(8, PH_IDX + 1)]):
    mm = s.meta
    print(f"[{i}] a={mm['a']:<24} x={mm['x']:<18} y(DST)={mm['y']:<18} z(SRC)={mm['z']}")
ph_inst = ph_insts[PH_IDX]
ph_g = PP.build_geometry(ph_inst, window=PH_WINDOW, span=PH_SPAN, include_pivot=PH_INCLUDE_PIVOT, i2_from=PH_I2_FROM,
                         res_frontier=PH_RES_FRONTIER, res_seed=PH_SEED, ctx=PH_CTX)
PH_CFG = (PH_DT, PH_CONN, PH_ORDER, PH_SRC_LANG, PH_DST_LANG, PH_SEED, PH_IDX, PH_SPAN, PH_WINDOW, PH_INCLUDE_PIVOT,
          PH_I2_FROM, PH_RES_FRONTIER)      # the run cell checks these still match
_ids = PP._ids(ph_g["dst_tok"])
_sw, _dw, _seg, _rseg = ph_g["src_w"], ph_g["dst_w"], ph_g["seg"], ph_g["res_seg"]
print(f"\nz (SRC) = {ph_g['c_desc']!r}\ny (DST) = {ph_g['d_desc']!r}\ne (p_res) = {ph_g['e_desc']!r} "
      f"({ph_g['e']}; {len(PP.res_candidates(ph_inst, ctx=PH_CTX))} allowed)")
print(f"window ({PH_WINDOW}, {PH_SPAN}): SRC {_sw[0]}..{_sw[-1]} -> DST {_dw[0]}..{_dw[-1]} ({len(_dw)} tok; pad "
      f"{ph_g['pad_src']}/{ph_g['pad_dst']}) · I2 window DST {_seg[0]}..{_seg[-1]} "
      f"({mt.tokenizer.decode([_ids[_seg[0]]])!r} .. {mt.tokenizer.decode([_ids[_seg[-1]]])!r}) == p_res {_rseg[0]}..{_rseg[-1]} "
      f"· pivot {mt.tokenizer.decode([_ids[-1]])!r}")
show_spans(ph_g["src_tok"], f"SRC · z={ph_inst.meta['z']} · transplanted window (orange)", kv_span=(_sw[0], _sw[-1] + 1), piv=-1)
show_spans(ph_g["dst_tok"], f"DST · y={ph_inst.meta['y']} · window (orange) · I2 window (yellow)",
           task_span=(_seg[0], _seg[-1] + 1), kv_span=(_dw[0], _dw[-1] + 1), piv=-1)
show_spans(ph_g["res_tok"], f"p_res · e={ph_g['e']} · I2 donor window (yellow)", task_span=(_rseg[0], _rseg[-1] + 1),
           piv=ph_g["res_tok"]["input_ids"].shape[1])

[0] a=count_ends_consonant     x=all_alpha          y(DST)=has_double         z(SRC)=capitalized
[1] a=total_length             x=same_reversed      y(DST)=ends_s             z(SRC)=all_alpha
[2] a=shortest_word_length     x=starts_a           y(DST)=same_reversed      z(SRC)=ends_s
[3] a=any_empty                x=vowel_start        y(DST)=all_lower          z(SRC)=length_five
[4] a=has_palindrome           x=long               y(DST)=starts_a           z(SRC)=all_lower
[5] a=count_all_lowercase      x=same_reversed      y(DST)=vowel_start        z(SRC)=long
[6] a=count_all_lowercase      x=starts_a           y(DST)=palindrome         z(SRC)=short
[7] a=count_starts_vowel       x=all_lower          y(DST)=contains_digit     z(SRC)=short

z (SRC) = 'keep only the words that start with an uppercase letter'
y (DST) = 'keep only the words that contain a repeated adjacent character'
e (p_res) = 'keep only the words that are shorter than four characters' (short; 8 allowed)
window (span, lis

╭─ SRC · z=capitalized · transplanted window (orange) ────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that end with a consonant\n                                                             │
│ (2) the list of words in `words` that consist only of letters and start with an uppercase letter\n              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[-1].isalpha() and w[-1].lower() not in "aeiou")\n                       │
│     b = [w for w in words if w.isalpha() and                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · y=has_double · window (orange) · I2 window (yellow) ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that end with a consonant\n                                                             │
│ (2) the list of words in `words` that consist only of letters and contain a repeated adjacent character\n       │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[-1].isalpha() and w[-1].lower() not in "aeiou")\n                       │
│     b = [w for w in words if w.isalpha() and                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ p_res · e=short · I2 donor window (yellow) ────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that end with a consonant\n                                                             │
│ (2) the list of words in `words` that consist only of letters and are shorter than four characters\n            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[-1].isalpha() and w[-1].lower() not in "aeiou")\n                       │
│     b = [w for w in words if w.isalpha() and                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# ── one run: sanity · clean · patched · p_res readout · oracle + rule verdict ──
assert PH_CFG == (PH_DT, PH_CONN, PH_ORDER, PH_SRC_LANG, PH_DST_LANG, PH_SEED, PH_IDX, PH_SPAN, PH_WINDOW, PH_INCLUDE_PIVOT,
                  PH_I2_FROM, PH_RES_FRONTIER), "knobs changed since the setup cell -- re-run it"
_L = ph_g["dst_tok"]["input_ids"].shape[1]
_layers = list(range(mt.n_layer))
_h0 = PP.layer_outputs(mt, ph_g["dst_tok"], list(range(_L)), _layers)
with PP.intervention_stack(ph_g, **PH_KW)():
    _h1 = PP.layer_outputs(mt, ph_g["dst_tok"], list(range(_L)), _layers)
_d = torch.stack([(_h1[l] - _h0[l]).abs().amax(-1) for l in _layers])
rich_print(f"on: I1={PH_I1} I2={PH_I2} I3={PH_I3} (V×{PH_I3_V_SCALE:g}, α={PH_ALPHA:g}) · max|Δresid| vs clean DST — window "
           f"{_d[:, ph_g['dst_w'][0]:ph_g['dst_w'][-1] + 1].max():.3g} · I2 window {_d[:, ph_g['seg']].max():.3g} · pivot {_d[:, -1].max():.3g}")

ph_clean = PP.clean_runs(ph_g, readout_heads=PH_READOUT_HEADS, max_new=PH_MAX_NEW, ctx=PH_CTX)
ph_pat = PP.run_patched(ph_g, readout_heads=PH_READOUT_HEADS, max_new=PH_MAX_NEW, **PH_KW)
ph_ro_res = PP.res_readout(ph_g, readout_heads=PH_READOUT_HEADS, ctx=PH_CTX) if PH_I2 and PH_I2_DONOR == "res" else None
ph_raw = dict(**ph_clean, **ph_pat, **({"ro_res": ph_ro_res} if ph_ro_res is not None else {}))
ph_rule = ({k: H.rule_verdict(ph_inst, ph_raw[f"gen_{k}"])["label"] for k in ("src", "dst", "patched")}
           if PH_DST_LANG == "python" and PH_SRC_LANG == "python" else
           {"patched": H.rule_verdict(ph_inst, ph_raw["gen_patched"])["label"]} if PH_DST_LANG == "python" else {})
for tag, col, k in (("clean SRC (z)", "red1", "src"), ("clean DST (y)", "sea_green3", "dst"),
                    (f"patched DST · I1={PH_I1} I2={PH_I2} I3={PH_I3} V×{PH_I3_V_SCALE:g}", "salmon1", "patched")):
    rich_print(Panel(Text(ph_raw[f"gen_{k}"]), title=f"[{col}]{escape(tag)}[/{col}] · rule={ph_rule.get(k, 'n/a')} · goalscope @ "
                     f"pivot: [italic]{escape(ph_raw[f'ro_{k}'])}[/italic]", title_align="left", border_style="grey42"))
if ph_ro_res is not None:
    rich_print(f"[grey58]clean p_res readout: {escape(ph_ro_res)}[/grey58]")
if PH_ORACLE:
    ph_causal, ph_readout, ph_readout_res = PP.judge(ph_inst, ph_raw, e_desc=ph_g["e_desc"])
    rich_print(f"[bold]eval[/bold] · causal={'HIT' if ph_causal['triggered'] else 'miss'}({ph_causal['label']}) · readout "
               f"{ph_readout['source']}>{ph_readout['dest']}>{ph_readout['patched']} "
               f"[{'valid' if ph_readout['valid'] else 'inv'},{'shift' if ph_readout['shifted'] else 'no'}]"
               + (f" · readout3={ph_readout_res['patched']} (p_res clean={ph_readout_res['res_clean']})" if ph_readout_res else ""))

on: I1=False I2=True I3=True (V×1, α=1) · max|Δresid| vs clean DST — window 88 · I2 window 256 · pivot 203

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:111: operator(): block: [5,0,0], thread: [0,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:111: operator(): block: [5,0,0], thread: [1,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:111: operator(): block: [5,0,0], thread: [2,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:111: operator(): block: [5,0,0], thread: [3,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:111: operator(): block: [5,0,0], thread: [4,0,0] Assertion `-sizes[i] <= index && index < sizes[i] && "index out of bounds"` failed.
/pytorch/aten/src/ATen/native/cuda/IndexKernel.cu:111: operator(): block: [5,0,0], th

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


### Compare configs on this instance

`PH_GRID` = list of (I2 on?, window, V) — each row reuses the instance, runs `PP.run_patched` (clean runs shared per token
sequence) and the oracle. Defaults: the batch grid (task span · I2 off / on · V×1 / ×4, and span + progress).

In [ ]:
PH_GRID = [(False, "span", 1.0), (True, "span", 1.0), (False, "span", 4.0), (True, "span", 4.0),
           (False, "span+progress", 1.0), (True, "span+progress", 1.0)]
_geoms, _cleans, _rows = {}, {}, []
for i2, win, vs in PH_GRID:
    if win not in _geoms:
        _geoms[win] = PP.build_geometry(ph_inst, window=win, span=PH_SPAN, include_pivot=PH_INCLUDE_PIVOT, i2_from=PH_I2_FROM,
                                        res_frontier=PH_RES_FRONTIER, res_seed=PH_SEED, ctx=PH_CTX)
    g = _geoms[win]
    ck = (tuple(PP._ids(g["src_tok"])), tuple(PP._ids(g["dst_tok"])))
    if ck not in _cleans:
        _cleans[ck] = PP.clean_runs(g, readout_heads=PH_READOUT_HEADS, max_new=PH_MAX_NEW, ctx=PH_CTX)
    pat = PP.run_patched(g, readout_heads=PH_READOUT_HEADS, max_new=PH_MAX_NEW, **dict(PH_KW, i2=i2, v_scale=vs))
    raw = dict(**_cleans[ck], **pat)
    if i2 and PH_I2_DONOR == "res":
        raw["ro_res"] = PP.res_readout(g, readout_heads=PH_READOUT_HEADS, ctx=PH_CTX)
    ev = PP.judge(ph_inst, raw, e_desc=g["e_desc"]) if PH_ORACLE else (None, None, None)
    rule = H.rule_verdict(ph_inst, raw["gen_patched"])["label"] if PH_DST_LANG == "python" else "n/a"
    _rows.append((i2, win, vs, raw, ev, rule))
t = Table(title=escape(f"multi-hierarchy path patching · {PH_DT} · {PH_CONN} · {PH_ORDER} · {PH_SRC_LANG}->{PH_DST_LANG} · "
                       f"instance {PH_IDX} · z={ph_inst.meta['z']} y={ph_inst.meta['y']} e={ph_g['e']}"), show_lines=True)
for c_ in ("I2", "window", "V", "first line (patched)", "rule", "causal", "readout c/d", "readout 3-way", "goalscope @ pivot"):
    t.add_column(c_, overflow="fold")
for i2, win, vs, raw, (ca, ro, rr), rule in _rows:
    t.add_row("on" if i2 else "off", win, f"×{vs:g}", escape(next((l for l in raw["gen_patched"].splitlines() if l.strip()), "")[:70]),
              rule, (f"{'HIT' if ca['triggered'] else 'miss'} ({ca['label']})" if ca else "—"),
              (f"{ro['patched']} [{'valid' if ro['valid'] else 'inv'}]" if ro else "—"), (rr["patched"] if rr else "—"),
              escape(raw["ro_patched"][:80]))
Console(width=220).print(t)